# 2d, Koszul bracket Leibniz: $[\omega, f\eta]_{T^*M} = f\,[\omega, \eta]_{T^*M} + (\pi^\sharp(\omega)f)\,\eta$

**Problem (d).** Right-Leibniz rule for the Koszul bracket on $T^*M$:

$$
[\omega, f\eta]_{T^*M} = f\,[\omega, \eta]_{T^*M} + \bigl(\pi^\sharp(\omega)\,f\bigr)\,\eta,
$$

The definition of the Koszul bracket:

$$
[\alpha, \beta]_K \;=\; \mathcal{L}_{\pi^\sharp\alpha}\beta \;-\; \mathcal{L}_{\pi^\sharp\beta}\alpha \;-\; d\langle \pi^\sharp\alpha, \beta \rangle.
$$

Since $\pi^\sharp$ is a tensor (C^∞-linear) and $\pi$ is anti-symmetric, $\langle\pi^\sharp\alpha,\beta\rangle + \langle\pi^\sharp\beta,\alpha\rangle = 0$. These two facts together close the right-Leibniz rule.

## Strategy, why qualitatively different from 2a/b/c

2a/b/c were **operator-level** identities (Cartan magic, $d^2=0$, pairing). 2d, on the other hand, demands **$C^\infty$-module algebra**: how the product $f\eta$ behaves under three distinct geometric operators:

| Term | Identity opened | Available in engine? |
|---|---|---|
| $\mathcal{L}_{X_\omega}(f\eta)$ | $X_\omega(f)\,\eta + f\,\mathcal{L}_{X_\omega}(\eta)$ | **Automatic**, Cartan magic + product-rule |
| $\mathcal{L}_{\pi^\sharp(f\eta)}\omega$ | $f\,\mathcal{L}_{X_\eta}(\omega) + df\wedge\iota_{X_\eta}(\omega)$ | **Missing**, $\pi^\sharp$ $C^\infty$-linearity + $\mathcal{L}_{fX}$ expansion |
| $d\langle X_\omega, f\eta\rangle$ | $df\cdot\langle X_\omega,\eta\rangle + f\cdot d\langle X_\omega,\eta\rangle$ | **Automatic**, $d$ Leibniz + pairing C^∞-linearity (axiom) |
| Closing $df\cdot\langle X_\omega,\eta\rangle$ with $df$ | $\iota_{X_\eta}(\omega) = -\langle X_\omega,\eta\rangle$ ($\pi$ anti-sym) | **Missing**, Musical compatibility rank-2 (Phase 12 #8) |

Two genuine geometric inputs: **(i)** $\pi^\sharp$ $C^\infty$-linearity + $\mathcal{L}_{fX}$ rule, **(ii)** $\pi$ anti-symmetry. We add both as inline axioms; the rest is closed by the engine.

**Mode discipline.** Cartan-mode $\mathcal{L}_X$ is opened to $d\circ\iota_X + \iota_X\circ d$ by the engine every time it sees one. We **want** this behavior for $X_\omega$ (so the LHS's $\mathcal{L}_{X_\omega}(f\eta)$ opens), but **not** for $X_\eta$ ($\mathcal{L}_{X_\eta}(\omega)$ should remain opaque, because $\eta$ and $\omega$ are generic 1-forms, Cartan expansion produces $d(\omega)$, $d(\eta)$, which are no longer reducible and the chain stalls). Solution: $X_\omega$ Cartan, $X_\eta$ flow.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Setup

- $f$, function (0-form).
- $\omega, \eta$, generic 1-forms.
- $X_\omega, X_\eta$, $\pi^\sharp(\omega)$, $\pi^\sharp(\eta)$ respectively. Since sharp is **a tensor (not a derivation)**, we must explicitly track the grading at the system level; for this reason we build $X_\omega$ and $X_\eta$ as **named `Derivation`s**, without keeping sharp itself as a $T^*M\to TM$ isomorphism (in the current framework `Sharp` is a Derivation atom, but its output degree is inconsistent when applied to a 1-form, see Phase 12 #6).
- $\mathcal{L}_{X_\omega}$ **cartan-mode**, $\mathcal{L}_{X_\eta}$ **flow-mode**.

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.calculus.lie_derivative import lie_derivative, LieDerivative
from jacopy.calculus.pairing import pairing, Pairing
from jacopy.core.expr import Expr, Integer, Neg, Product, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
omega = Symbol("ω")
eta = Symbol("η")
reg.declare(f, Graded(degree=0))
reg.declare(omega, Graded(degree=1))
reg.declare(eta, Graded(degree=1))

# π^♯(ω) and π^♯(η) - vector fields (degree-0 derivations).
X_om = Derivation("X_ω", degree=0)
X_et = Derivation("X_η", degree=0)

L_Xom = lie_derivative(X_om, definition="cartan")
L_Xet = lie_derivative(X_et, definition="flow")

# Build the Koszul bracket as a named Symbol; its definition is added as an axiom below.
koszul_om_feta = Symbol("[ω,fη]_K")
koszul_om_eta  = Symbol("[ω,η]_K")
reg.declare(koszul_om_feta, Graded(degree=1))
reg.declare(koszul_om_eta,  Graded(degree=1))

print("f, ω, η        :", f, omega, eta)
print("X_ω (cartan)   :", X_om, " L =", L_Xom)
print("X_η (flow)     :", X_et, " L =", L_Xet)
print("[ω,fη]_K       :", koszul_om_feta)
print("[ω,η]_K        :", koszul_om_eta)

f, ω, η        : f ω η
X_ω (cartan)   : X_ω  L = L_X_ω
X_η (flow)     : X_η  L = L_X_η
[ω,fη]_K       : [ω,fη]_K
[ω,η]_K        : [ω,η]_K


## 2. Axioms (4 of them)

**A-K1, `[ω, fη]_K` defining (with sharp $C^\infty$-linearity folded).** The pure Koszul definition would be $\mathcal{L}_{X_\omega}(f\eta) - \mathcal{L}_{\pi^\sharp(f\eta)}\omega - d\langle X_\omega, f\eta\rangle$; since $\pi^\sharp(f\eta) = f\cdot X_\eta$ ($C^\infty$-linearity) and $\mathcal{L}_{fX}\omega = f\,\mathcal{L}_X\omega + df\wedge\iota_X\omega$, we write the middle term in advance with this expanded form:

$$
[\omega, f\eta]_K \;\to\; \mathcal{L}_{X_\omega}(f\eta) - f\,\mathcal{L}_{X_\eta}(\omega) - df\wedge\iota_{X_\eta}(\omega) - d\langle X_\omega, f\eta\rangle.
$$

This approach is a practical necessity since the system does not recognize $\pi^\sharp$ as $C^\infty$-linear. When **Phase 12** lands (#6 multilinear evaluation, #8 musical compatibility bilinear), this folded form can be split into two separate lines (sharp linearity + $L_{fX}$).

**A-K2, `[ω, η]_K` defining.** Standard Koszul:

$$
[\omega, \eta]_K \;\to\; \mathcal{L}_{X_\omega}(\eta) - \mathcal{L}_{X_\eta}(\omega) - d\langle X_\omega, \eta\rangle.
$$

**A-pairing, pairing $C^\infty$-linear.** $\langle X_\omega, f\eta\rangle = f\,\langle X_\omega, \eta\rangle$.

**A-π-antisym, $\pi$ anti-symmetry.** $\iota_{X_\eta}(\omega) = -\langle X_\omega, \eta\rangle$. Because $\iota_{\pi^\sharp\eta}\omega = \omega(\pi^\sharp\eta) = \pi(\eta,\omega) = -\pi(\omega,\eta) = -\langle\pi^\sharp\omega, \eta\rangle = -\langle X_\omega,\eta\rangle$.

In [3]:
class KoszulDef_om_feta(Definition):
    """A-K1: [ω, fη]_K defining (sharp C∞-linearity + L_{fX} folded)."""
    name = "[ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩"

    def matches(self, expr):
        return expr == koszul_om_feta

    def rewrite(self, expr):
        return Sum(
            Act(L_Xom, Product(f, eta)),
            Neg(Product(f, Act(L_Xet, omega))),
            Neg(Product(Act(d, f), Act(interior(X_et), omega))),
            Neg(Act(d, pairing(X_om, Product(f, eta)))),
        )


class KoszulDef_om_eta(Definition):
    """A-K2: [ω, η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩."""
    name = "[ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩"

    def matches(self, expr):
        return expr == koszul_om_eta

    def rewrite(self, expr):
        return Sum(
            Act(L_Xom, eta),
            Neg(Act(L_Xet, omega)),
            Neg(Act(d, pairing(X_om, eta))),
        )


class PairingXomFeta(Definition):
    """A-pairing: ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩."""
    name = "⟨X_ω, fη⟩ = f·⟨X_ω, η⟩"

    def matches(self, expr):
        return (
            isinstance(expr, Pairing)
            and expr.alpha == X_om
            and expr.X == Product(f, eta)
        )

    def rewrite(self, expr):
        return Product(f, pairing(X_om, eta))


class PiAntiSymm(Definition):
    """A-π-antisym: ι_{X_η}(ω) = −⟨X_ω, η⟩."""
    name = "ι_{X_η}(ω) = −⟨X_ω, η⟩"

    def matches(self, expr):
        return (
            isinstance(expr, Act)
            and isinstance(expr.op, InteriorProduct)
            and expr.op.vector_field == X_et
            and expr.arg == omega
        )

    def rewrite(self, expr):
        return Neg(pairing(X_om, eta))


for axiom in (KoszulDef_om_feta, KoszulDef_om_eta, PairingXomFeta, PiAntiSymm):
    print("-", axiom.name)

- [ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩
- [ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩
- ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩
- ι_{X_η}(ω) = −⟨X_ω, η⟩


## 3. Engine

`default_engine` already carries Cartan magic, $d^2=0$, pairing ($\iota_X(df) = X(f)$), $\iota_X(f)=0$; product-rule ($d$ and $\iota$ Leibniz) is also automatic in the canonical-form pipeline. We add the four problem axioms on top.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(KoszulDef_om_feta())
engine.register(KoszulDef_om_eta())
engine.register(PairingXomFeta())
engine.register(PiAntiSymm())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 12 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - [ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩
 - [ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩
 - ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩
 - ι_{X_η}(ω) = −⟨X_ω, η⟩


## 4. Goal and proof

$$
[\omega, f\eta]_K \;=\; f\,[\omega, \eta]_K + (X_\omega(f))\,\eta.
$$

The $X_\omega(f) = (\pi^\sharp\omega)(f)$ on the RHS is a Hamiltonian-type scalar function (a vector field applied to a function).

In [5]:
lhs = koszul_om_feta
rhs = Sum(
    Product(f, koszul_om_eta),
    Product(Act(X_om, f), eta),
)

print("LHS:", lhs)
print("RHS:", rhs)

chain = ExpandAndSimplify().prove(
    lhs, rhs, registry=reg, engine=engine
)
print(f"\nCLOSED, {len(chain)} steps.")

LHS: [ω,fη]_K
RHS: ((f * [ω,η]_K) + (X_ω(f) * η))

CLOSED, 13 steps.


## 5. Proof chain, LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

{\allowdisplaybreaks\scriptsize
\begin{gather*}
[\omega,f\eta]_K \to L_{X_\omega}\!\left(f \, \eta\right) - f \, L_{X_\eta}\!\left(\omega\right) - d\!\left(f\right) \, \iota_{X_\eta}\!\left(\omega\right) - d\!\left(\langle X_\omega,\, f \, \eta \rangle\right) \quad \text{[[\ensuremath{\omega},f\ensuremath{\eta}]\_K = L\_X\ensuremath{\omega}(f\ensuremath{\eta}) \ensuremath{-} f\ensuremath{\cdot}L\_X\ensuremath{\eta}(\ensuremath{\omega}) \ensuremath{-} df\ensuremath{\cdot}\ensuremath{\iota}\_X\ensuremath{\eta}(\ensuremath{\omega}) \ensuremath{-} d\ensuremath{\langle}X\_\ensuremath{\omega}, f\ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)} \\
L_{X_\omega}\!\left(f \, \eta\right) \to \left(d \, \iota_{X_\omega}\right)\!\left(f \, \eta\right) + \left(\iota_{X_\omega} \, d\right)\!\left(f \, \eta\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\iota_{X_\eta}\!\left(\omega\right) \to -\langle X_\omega,\, \eta \rangle \quad \text{[\ensuremath{\iota}\_{X\_\ensuremath{\eta}}(\ensuremath{\omega}) = \ensuremath{-}\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)} \\
\langle X_\omega,\, f \, \eta \rangle \to f \, \langle X_\omega,\, \eta \rangle \quad \text{[\ensuremath{\langle}X\_\ensuremath{\omega}, f\ensuremath{\eta}\ensuremath{\rangle} = f\ensuremath{\cdot}\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)} \\
[\omega,\eta]_K \to L_{X_\omega}\!\left(\eta\right) - L_{X_\eta}\!\left(\omega\right) - d\!\left(\langle X_\omega,\, \eta \rangle\right) \quad \text{[[\ensuremath{\omega},\ensuremath{\eta}]\_K = L\_X\ensuremath{\omega}(\ensuremath{\eta}) \ensuremath{-} L\_X\ensuremath{\eta}(\ensuremath{\omega}) \ensuremath{-} d\ensuremath{\langle}X\_\ensuremath{\omega}, \ensuremath{\eta}\ensuremath{\rangle}]\,(axiom)} \\
L_{X_\omega}\!\left(\eta\right) \to \left(d \, \iota_{X_\omega}\right)\!\left(\eta\right) + \left(\iota_{X_\omega} \, d\right)\!\left(\eta\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\left(\left(\left(d \, \iota_{X_\omega}\right)\!\left(f \, \eta\right) + \left(\iota_{X_\omega} \, d\right)\!\left(f \, \eta\right)\right) - f \, L_{X_\eta}\!\left(\omega\right) - d\!\left(f\right) \, \left(-\langle X_\omega,\, \eta \rangle\right) - d\!\left(f \, \langle X_\omega,\, \eta \rangle\right)\right) - \left(f \, \left(\left(\left(d \, \iota_{X_\omega}\right)\!\left(\eta\right) + \left(\iota_{X_\omega} \, d\right)\!\left(\eta\right)\right) - L_{X_\eta}\!\left(\omega\right) - d\!\left(\langle X_\omega,\, \eta \rangle\right)\right) + X_\omega\!\left(f\right) \, \eta\right) \to \left(\left(\left(d\!\left(\iota_{X_\omega}\!\left(f\right)\right) \, \eta - \iota_{X_\omega}\!\left(f\right) \, d\!\left(\eta\right) + d\!\left(f\right) \, \iota_{X_\omega}\!\left(\eta\right) + f \, d\!\left(\iota_{X_\omega}\!\left(\eta\right)\right)\right) + \left(\iota_{X_\omega}\!\left(d\!\left(f\right)\right) \, \eta - d\!\left(f\right) \, \iota_{X_\omega}\!\left(\eta\right) + \iota_{X_\omega}\!\left(f\right) \, d\!\left(\eta\right) + f \, \iota_{X_\omega}\!\left(d\!\left(\eta\right)\right)\right)\right) - f \, L_{X_\eta}\!\left(\omega\right) - d\!\left(f\right) \, \left(-\langle X_\omega,\, \eta \rangle\right) - \left(d\!\left(f\right) \, \langle X_\omega,\, \eta \rangle + f \, d\!\left(\langle X_\omega,\, \eta \rangle\right)\right)\right) - \left(f \, \left(\left(d\!\left(\iota_{X_\omega}\!\left(\eta\right)\right) + \iota_{X_\omega}\!\left(d\!\left(\eta\right)\right)\right) - L_{X_\eta}\!\left(\omega\right) - d\!\left(\langle X_\omega,\, \eta \rangle\right)\right) + X_\omega\!\left(f\right) \, \eta\right) \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\iota_{X_\omega}\!\left(f\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_{X

## 6. Step by step

The flow of the chain:

1. **A-K1**, LHS expansion: $[\omega,f\eta]_K \to L_{X_\omega}(f\eta) - f\,L_{X_\eta}(\omega) - df\,\iota_{X_\eta}(\omega) - d\langle X_\omega, f\eta\rangle$.
2. **Cartan magic**, $L_{X_\omega}(f\eta) \to (d\iota_{X_\omega} + \iota_{X_\omega} d)(f\eta)$.
3. **A-π-antisym**, $\iota_{X_\eta}(\omega) \to -\langle X_\omega,\eta\rangle$.
4. **A-pairing**, $\langle X_\omega, f\eta\rangle \to f\cdot\langle X_\omega,\eta\rangle$.
5. **A-K2**, expansion of $[\omega,\eta]_K$ on the RHS.
6. **Cartan magic**, expansion of $L_{X_\omega}(\eta)$.
7. **product-rule**, large expansion: $d$ and $\iota$ Leibniz applied across all $f\cdot(\ldots)$ products; canonical form emerges.
8-11. **$\iota_X(f)=0$ + pairing**, $\iota_{X_\omega}(f)$ is zero, $\iota_{X_\omega}(df) = X_\omega(f)$.
12. **product-rule**, clearing of zeros.
13. **simplify**, canonical-form closure: all terms either cancel or drop to $0$.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] [ω,fη]_K = L_Xω(fη) − f·L_Xη(ω) − df·ι_Xη(ω) − d⟨X_ω, fη⟩ [axiom]
    [ω,fη]_K
 -> (L_X_ω((f * η)) + (-(f * L_X_η(ω))) + (-(d(f) * ι_X_η(ω))) + (-d(⟨X_ω, (f * η)⟩)))

[2] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_ω((f * η))
 -> ((d * ι_X_ω)((f * η)) + (ι_X_ω * d)((f * η)))

[3] ι_{X_η}(ω) = −⟨X_ω, η⟩ [axiom]
    ι_X_η(ω)
 -> (-⟨X_ω, η⟩)

[4] ⟨X_ω, fη⟩ = f·⟨X_ω, η⟩ [axiom]
    ⟨X_ω, (f * η)⟩
 -> (f * ⟨X_ω, η⟩)

[5] [ω,η]_K = L_Xω(η) − L_Xη(ω) − d⟨X_ω, η⟩ [axiom]
    [ω,η]_K
 -> (L_X_ω(η) + (-L_X_η(ω)) + (-d(⟨X_ω, η⟩)))

[6] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_ω(η)
 -> ((d * ι_X_ω)(η) + (ι_X_ω * d)(η))

[7] product-rule 
    ((((d * ι_X_ω)((f * η)) + (ι_X_ω * d)((f * η))) + (-(f * L_X_η(ω))) + (-(d(f) * (-⟨X_ω, η⟩))) + (-d((f * ⟨X_ω, η⟩)))) + (-((f * (((d * ι_X_ω)(η) + (ι_X_ω * d)(η)) + (-L_X_η(ω)) + (-d(⟨X_ω, η⟩)))) + (X_ω(f) * η))))
 -> (((((d(ι_X_ω(f)) * η) + (-(ι_X_ω(f) * d(η))) + (d(f) * ι_X_ω(η)) + (f * d(ι_X_ω(η)))) + ((ι_X_ω(d(f)) * η) + (-

## Conclusion

$$
\boxed{\;[\omega, f\eta]_K = f\,[\omega, \eta]_K + (\pi^\sharp(\omega)f)\,\eta.\;}
$$

**13-step chain, 4 inline axioms.** Qualitative difference from 2a/b/c: 2d demands $C^\infty$-module algebra, sharp tensoriality, $L_{fX}$ expansion, $\pi$ anti-symmetry. These three are **genuine geometric inputs**, things the framework cannot know.

**Infrastructure fix that came out of this pass:** `jacopy/algorithms/sort_product.py:_degree_of` could not extract the degree of compound factors like `Act(d, f)` (only Scalar/Derivation/registry-Graded). The fix added in this pass falls back to `algebra.derivation.degree_of`; the canonical-form pipeline now sorts compound factors smoothly. Without this, the 2d notebook would not run without a monkey-patch.

**Phase 12 tags.** Ideally the following three pieces will simplify this notebook further:

- **#6 Multilinear evaluation + sharp linearity**, the axiom $\pi^\sharp(f\eta) = f\cdot\pi^\sharp(\eta)$ becomes built-in, splitting A-K1's folded form into two separate lines.
- **#8 Musical compatibility bilinear**, reduces A-π-antisym to a theorem.
- **12.C(c) `register_hamiltonian_defining_relation`** + Sharp/Pairing $C^\infty$-linearity wrappers, collapses A-K1, A-K2, A-pairing to a one-line API.

Until these three land, the 4-axiom pattern remains standard for Koszul-family notebooks.